# P0 · Review follow-ups: paired intervals, defect vocabulary, 3CAD bootstrap (exploratory)

| Section | Question |
|---|---|
| Paired intervals for the controls | are the medoid, trimming, large-M and σ/FPR differences larger than category-to-category noise? |
| Defect vocabulary | do words mean consistent things (position, appearance), and do defects of one type trigger the same words? Codebook vs. the same-size coreset used as a vocabulary |
| Example figures | for each defect type: the defect, and normal examples of its signature word |
| 3CAD image-level bootstrap | the two-level bootstrap from notebook 09, on 3CAD |

Reuses cached MVTec features and vocabularies. The figures need the MVTec images and the 3CAD bootstrap needs
the 3CAD archive (both already on Drive). **Runtime (T4):** about 1–1.5 h. Finished runs are saved;
Run all again after a disconnect to resume.

## 1 · Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: DINOv2 extraction will be slow"

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Get the code. Option A: GitHub (default). Option B: set REPO_URL = "" to install
# from bovw-forensics.zip uploaded to MyDrive/bovw-forensics/.
REPO_URL = "https://github.com/M3R2T2L2/bovw_forensics.git"
ZIP_PATH = "/content/drive/MyDrive/bovw-forensics/bovw-forensics.zip"

import os, shutil, subprocess, zipfile
shutil.rmtree("/content/bovw_forensics", ignore_errors=True)
if REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/bovw_forensics"], check=True)
else:
    zipfile.ZipFile(ZIP_PATH).extractall("/content/")
    os.rename("/content/bovw-forensics", "/content/bovw_forensics")

%cd /content/bovw_forensics
!pip install -q -e ".[dev]"   # torch, torchvision, transformers come preinstalled on Colab

In [ ]:
# 30-second check that the torch-free core works in this runtime
!python -m pytest -q

In [ ]:
# STL-10 on Drive: restore it here (skips the ~45 min download), or save it after the first download.
import os, shutil
DRIVE_STL = "/content/drive/MyDrive/bovw-forensics/data/stl10_binary"
LOCAL_STL = "/content/data/stl10_binary"
if os.path.isdir(DRIVE_STL) and not os.path.isdir(LOCAL_STL):
    shutil.copytree(DRIVE_STL, LOCAL_STL); print("Restored STL-10 from Drive")
elif os.path.isdir(LOCAL_STL) and not os.path.isdir(DRIVE_STL):
    shutil.copytree(LOCAL_STL, DRIVE_STL); print("Saved STL-10 to Drive for future runtimes")
else:
    print("STL-10 will download on first use; re-run this cell afterwards to save it to Drive"
          if not os.path.isdir(LOCAL_STL) else "STL-10 already local and on Drive")

## 2 · Data
The archive is kept on Drive so it downloads only once; each new runtime extracts it locally (~3–5 min).

In [ ]:
import os, subprocess
from bovw import ad_data
DRIVE_TAR = "/content/drive/MyDrive/bovw-forensics/data/mvtec_anomaly_detection.tar.xz"
LOCAL_ROOT = "/content/mvtec"

# Set to True only after accepting the licence on the MVTec page above.
I_ACCEPT_MVTEC_LICENCE = False

if not os.path.isfile(DRIVE_TAR) or os.path.getsize(DRIVE_TAR) < 1e9:   # missing, or a stub from a failed download
    if not I_ACCEPT_MVTEC_LICENCE:
        raise SystemExit("Accept the MVTec AD licence, then set I_ACCEPT_MVTEC_LICENCE = True "
                         "(or upload mvtec_anomaly_detection.tar.xz to MyDrive/bovw-forensics/data/).")
# Downloads only if missing or corrupt; verifies the SHA-256 (hashing ~5 GB takes about a minute).
ad_data.ensure_archive(DRIVE_TAR)

if not os.path.isdir(f"{LOCAL_ROOT}/bottle"):
    os.makedirs(LOCAL_ROOT, exist_ok=True)
    subprocess.run(["tar", "-xf", DRIVE_TAR, "-C", LOCAL_ROOT], check=True)
print(sorted(os.listdir(LOCAL_ROOT)))

## 3 · Paired category-level intervals for the controls (MVTec AD, VisA)

In [ ]:
import pandas as pd
from bovw import stats

RES = "/content/drive/MyDrive/bovw-forensics/results"
CACHE = "/content/drive/MyDrive/bovw-forensics/cache"
ctrl = {ds: pd.read_csv(f"{RES}/p0_review_{ds}.csv") for ds in ["mvtec", "visa"]}
main = ["image_auroc", "image_auroc_top10", "pixel_auroc", "aupro"]
extra = ["aupro_f5", "aupro_f10", "pixel_auroc_s0", "aupro_s0", "pixel_auroc_s8", "aupro_s8"]
pairs = [("medoid_knn", "codebook_dist", (64, 256, 1024), main + extra),
         ("medoid_knn", "coreset_knn", (64, 256, 1024), main + extra),
         ("coreset_trim1", "coreset_knn", (64, 256, 1024, 4096, 16384), main),
         ("coreset_trim5", "coreset_knn", (64, 256, 1024, 4096, 16384), main),
         ("codebook_gpu", "coreset_knn", (1024, 4096, 16384), main),
         ("codebook_gpu", "codebook_dist", (1024,), main),
         ("codebook_dist", "coreset_knn", (64, 1024), extra)]
paired = pd.concat([stats.compare(ctrl, a, b, ks=ks, metrics=m) for a, b, ks, m in pairs], ignore_index=True)
paired.to_csv(f"{RES}/p0_review_controls_paired.csv", index=False)
show = paired[paired.dataset == "pooled"].drop(columns=["p_sign", "p_wilcoxon"])
show.round(3).to_string(max_rows=200)

## 4 · Is the vocabulary a usable defect language? (MVTec AD)

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU: Runtime > Change runtime type > T4 GPU, then Run all.")
from bovw.sweep import load_config
from bovw import lexicon

lcfg = load_config("configs/p0_lexicon_mvtec.yaml")
lcfg["data"]["root"] = LOCAL_ROOT
lex = lexicon.run(lcfg)
lsum = lexicon.summary(lex)
lsum.to_csv(f"{RES}/p0_lexicon_mvtec_summary.csv")
lsum.round(3)

In [ ]:
# Paired codebook - coreset differences over categories (seeds averaged), with bootstrap CIs
lx = lex.rename(columns={"memory": "method"})
lpaired = stats.compare({"mvtec": lx}, "codebook", "coreset", ks=(64, 256, 1024),
                        metrics=["position_nmi_gain", "cohesion", "knn1", "knn5", "sig_cover", "sig_false"])
lpaired.to_csv(f"{RES}/p0_lexicon_mvtec_paired.csv", index=False)
lpaired.drop(columns=["p_sign", "p_wilcoxon"]).round(3)

## 5 · Example figures: each defect type and its signature word (K = 256, seed 0)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from bovw import ad_data, cache as fcache
from bovw.anomaly import _norm, cached_vocabulary, grid_side
from bovw.vocab import build_vocabulary

K, SEED = 256, 0
vcfg = {"max_descriptors": 200000, "spherical": True}
for cat in ["cable", "screw", "metal_nut", "hazelnut"]:
    tag = f"mvtec_{cat}_s448_m256"
    tr, _ = fcache.get_or_compute(CACHE, f"{tag}_train", "dinov2_s", {"image_size": 448}, lambda: None)
    te, _ = fcache.get_or_compute(CACHE, f"{tag}_test", "dinov2_s", {"image_size": 448}, lambda: None)
    trd, ted = _norm(tr.desc, True), _norm(te.desc, True)
    vkey = {"tag": tag, "extractor": "dinov2_s", "params": {"image_size": 448}, "normalize": True,
            "k": K, "seed": SEED, "vocab": vcfg}
    vocab, _ = cached_vocabulary(CACHE, vkey, lambda: build_vocabulary(trd, K, seed=SEED, **vcfg))
    tr_w, te_w, te_d = lexicon.memory_words("codebook", trd, ted, K, SEED, vocab)
    split = ad_data.load_mvtec(LOCAL_ROOT, cat, 448, 256)
    types = np.array(split.test_types)
    anom = (np.asarray(split.test_labels) == 1) & (types != "combined")
    fig = lexicon.defect_vocabulary_figure(split.train_images, split.test_images, tr_w, te_w, te_d, types, anom,
                                           grid_side(te), K, title=f"MVTec {cat}: defect type -> signature word (K = {K})")
    fig.savefig(f"{RES}/p0_lexicon_{cat}_K{K}.png", dpi=130, bbox_inches="tight")
    plt.show()

## 6 · Image-level bootstrap on 3CAD
Needs only the test labels. They are rebuilt once from the 3CAD archive in `MyDrive/bovw-forensics/data/`
and saved to Drive, so later runs need no archive. If the archive is not on Drive, this section is skipped
(download it in a browser from the link in notebook 06 and upload it there).

In [ ]:
import os, glob, shutil, subprocess
from bovw import ad_data

def get_mvtec():
    tar, local = "/content/drive/MyDrive/bovw-forensics/data/mvtec_anomaly_detection.tar.xz", "/content/mvtec"
    if not os.path.isfile(tar):
        raise SystemExit("Put mvtec_anomaly_detection.tar.xz in MyDrive/bovw-forensics/data/ (see notebook 02).")
    if not os.path.isdir(f"{local}/bottle"):
        os.makedirs(local, exist_ok=True)
        subprocess.run(["tar", "-xf", tar, "-C", local], check=True)
    return local

def get_visa():
    tar, local = "/content/drive/MyDrive/bovw-forensics/data/VisA_20220922.tar", "/content/visa"
    if not os.path.isfile(tar) or os.path.getsize(tar) < 1e9:
        ad_data.ensure_archive(tar, url=ad_data.VISA_URL, sha256=None)
    if not os.path.isdir(local):
        os.makedirs(local)
        subprocess.run(["tar", "-xf", tar, "-C", local], check=True)
    return ad_data.find_visa_root(local)

def get_3cad():
    local, archive = "/content/3cad", "/content/3cad_archive"
    if not os.path.isdir(local):
        copy = glob.glob("/content/drive/MyDrive/bovw-forensics/data/3CAD*")
        if not copy:
            raise SystemExit("Put the 3CAD archive in MyDrive/bovw-forensics/data/ (see notebook 06).")
        shutil.copy(copy[0], archive)
        os.makedirs(local)
        magic = open(archive, "rb").read(4)
        fmt = "zip" if magic[:2] == b"PK" else ("gztar" if magic[:2] == b"\x1f\x8b" else "tar")
        shutil.unpack_archive(archive, local, fmt)
        os.remove(archive)
    return ad_data.find_root(local, "Copper_Stator")

from pathlib import Path
from bovw import review
from bovw.anomaly import _LazyAD

META = Path(f"{CACHE}/3cad_meta")            # labels only (small); 3CAD features were never on Drive
cats = ad_data.categories("3cad")
need = [c for c in cats if not (META / f"3cad_{c}_s448_m256__meta.npz").exists()]
ok = True
if need:
    try:
        ROOT3 = get_3cad()
        for cat in need:
            _LazyAD({"name": "3cad", "root": ROOT3, "size": 448, "mask_size": 256}, cat, META,
                    f"3cad_{cat}_s448_m256").get_meta()
    except SystemExit as e:
        ok = False
        print("Skipping the 3CAD bootstrap:", e)
if ok:
    specs = [{"label": "3cad: codebook - coreset", "cfg_name": "p0_anomaly_3cad", "dataset": "3cad", "categories": cats,
              "a": ("codebook_dist", k), "b": ("coreset_knn", k)} for k in [64, 256, 1024]]
    specs += [{"label": "3cad: hybrid75 - codebook", "cfg_name": "p0_hybrid_3cad", "dataset": "3cad", "categories": cats,
               "a": ("hybrid_w75", k), "b": ("codebook_dist", k)} for k in [256, 1024]]
    img3 = review.image_level_table(specs, results_dir=RES, cache_dir=META, n_boot=2000)
    img3.to_csv(f"{RES}/p0_review_image_bootstrap_3cad.csv", index=False)
    display(img3.round(4))

## 7 · Record the environment (pinned package versions for the archive)

In [ ]:
import platform, subprocess, torch
lock = subprocess.run(["pip", "freeze"], capture_output=True, text=True).stdout
head = (f"# Colab environment, {pd.Timestamp.now():%Y-%m-%d}; Python {platform.python_version()}; "
        f"torch {torch.__version__}; CUDA {torch.version.cuda}; GPU {torch.cuda.get_device_name(0)}\n")
open(f"{RES}/requirements-colab.lock", "w").write(head + lock)
print(head, len(lock.splitlines()), "packages written to requirements-colab.lock on Drive")